# VQE Ground-State Estimation

**Learning objectives**

- Define a Hamiltonian
- Optimize a parameterized ansatz
- Compare variational and exact energies

## Environment setup



In [ ]:
# Run once in a fresh Colab/Jupyter environment, then restart the kernel if prompted.
%pip install -q qiskit qiskit-aer matplotlib pylatexenc scipy sympy

In [ ]:
from importlib.metadata import version
for package in ["qiskit", "qiskit-aer", "matplotlib", "pylatexenc", "scipy", "sympy"]:
    print(f"{package:15}: {version(package)}")

## One-qubit VQE

For $H=0.5Z-1.0X$, minimize $\langle H\rangle$ using a parameterized quantum state.

In [ ]:
import numpy as np
from scipy.optimize import minimize
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, SparsePauliOp
H=SparsePauliOp.from_list([("Z",0.5),("X",-1.0)])
def ansatz(x):
    qc=QuantumCircuit(1); qc.ry(x[0],0); qc.rz(x[1],0); return qc
def energy(x): return float(np.real(Statevector.from_instruction(ansatz(x)).expectation_value(H)))
history=[]
def objective(x):
    e=energy(x); history.append(e); return e
res=minimize(objective,[0.2,0.1],method="COBYLA",options={"maxiter":150,"tol":1e-9})
exact=np.linalg.eigvalsh(H.to_matrix())[0]
print("parameters",res.x,"VQE",res.fun,"exact",exact,"error",abs(res.fun-exact))
display(ansatz(res.x).draw("mpl")); display(Statevector.from_instruction(ansatz(res.x)).draw("latex"))

## Convergence plot

The variational principle keeps the estimated energy at or above the exact ground-state energy.

In [ ]:
import matplotlib.pyplot as plt
plt.plot(history,marker="."); plt.axhline(exact,color="red",linestyle="--",label="exact")
plt.xlabel("Function evaluation"); plt.ylabel("Energy"); plt.title("VQE convergence"); plt.legend(); plt.grid(); plt.show()